
# Skin Tone Classification — Training Notebook (Monk Skin Tone Scale)

Production-oriented training pipeline for a CNN skin-tone classifier, built around
the **STW-style** dataset (composite of CelebA / LFW / CASIA Face Africa / CASIA Face V5 /
FEI / Faces94-95 / FERET, annotated on the 10-point Monk Skin Tone scale).

## Expected data layout

This notebook uses the standard `ImageFolder` layout. Build your STW composite into this
structure before running (one folder per Monk-tone class, 1..10 or however many you use):

```
data/
├── train/
│   ├── monk_01/
│   ├── monk_02/
│   ├── ...
│   └── monk_10/
├── val/
│   ├── monk_01/
│   └── ...
└── test/
    ├── monk_01/
    └── ...
```

If you only have a single flat folder per class (no pre-made split), see the
**"Optional: auto-split a flat dataset"** cell below — it will create train/val/test
folders for you with stratified sampling.

## What this notebook does
1. Reproducible data pipeline with augmentation + class-imbalance handling
2. Transfer-learning CNN (configurable backbone) with staged unfreezing
3. Mixed-precision training, LR scheduling, early stopping, best-checkpoint saving
4. Evaluation: per-class metrics, confusion matrix, **fairness check across tone groups**
5. Inference helper
6. Production export: TorchScript + ONNX


In [ ]:

# If running in a fresh environment, uncomment:
# %pip install torch torchvision scikit-learn pandas numpy matplotlib pillow tqdm onnx onnxruntime -q


In [ ]:

import os
import json
import time
import random
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
)
from tqdm.auto import tqdm

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## Config

All knobs live here. Nothing else in the notebook should need hardcoded values.

In [ ]:

@dataclass
class Config:
    # --- data ---
    data_dir: str = "data"                 # expects data/train, data/val, data/test
    class_names: list = field(default_factory=lambda: [f"monk_{i:02d}" for i in range(1, 11)])
    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 4
    use_weighted_sampler: bool = True       # handle class imbalance at the sampler level

    # --- model ---
    backbone: str = "efficientnet_b0"       # one of: resnet50, efficientnet_b0, convnext_tiny
    pretrained: bool = True
    freeze_backbone_epochs: int = 3         # train only the head for N epochs, then unfreeze
    dropout: float = 0.3

    # --- optimization ---
    epochs: int = 30
    lr_head: float = 3e-4
    lr_backbone: float = 3e-5
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    grad_clip_norm: float = 1.0
    early_stopping_patience: int = 7

    # --- misc ---
    seed: int = 42
    output_dir: str = "outputs"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @property
    def num_classes(self):
        return len(self.class_names)


cfg = Config()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(cfg)


In [ ]:

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)



## Optional: auto-split a flat dataset

Skip this cell if `data/train`, `data/val`, `data/test` already exist. If you only have
one folder per class (e.g. from merging STW sources), this creates a stratified 70/15/15
split without duplicating files on disk (uses symlinks; falls back to copy on Windows).


In [ ]:

def stratified_split(flat_dir, out_dir, split=(0.7, 0.15, 0.15), seed=42):
    flat_dir, out_dir = Path(flat_dir), Path(out_dir)
    assert abs(sum(split) - 1.0) < 1e-6
    rng = random.Random(seed)

    class_dirs = [d for d in flat_dir.iterdir() if d.is_dir()]
    for split_name in ["train", "val", "test"]:
        for cls_dir in class_dirs:
            (out_dir / split_name / cls_dir.name).mkdir(parents=True, exist_ok=True)

    for cls_dir in class_dirs:
        files = [f for f in cls_dir.iterdir() if f.is_file()]
        rng.shuffle(files)
        n = len(files)
        n_train = int(n * split[0])
        n_val = int(n * split[1])
        buckets = {
            "train": files[:n_train],
            "val": files[n_train:n_train + n_val],
            "test": files[n_train + n_val:],
        }
        for split_name, split_files in buckets.items():
            for f in split_files:
                dest = out_dir / split_name / cls_dir.name / f.name
                if not dest.exists():
                    try:
                        os.symlink(f.resolve(), dest)
                    except OSError:
                        shutil.copy2(f, dest)
    print(f"Split written to {out_dir}")

# Example usage — uncomment and point at your flat, unsplit dataset:
# stratified_split("data_flat", cfg.data_dir, seed=cfg.seed)


## Transforms & Datasets

In [ ]:

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((cfg.img_size + 32, cfg.img_size + 32)),
    transforms.RandomCrop(cfg.img_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.02),
    # Keep hue jitter small and deliberate — this is a color-sensitive task,
    # so aggressive hue/saturation augmentation can corrupt the label signal.
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

eval_transform = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(Path(cfg.data_dir) / "train", transform=train_transform)
val_ds = datasets.ImageFolder(Path(cfg.data_dir) / "val", transform=eval_transform)
test_ds = datasets.ImageFolder(Path(cfg.data_dir) / "test", transform=eval_transform)

# Sanity check: folder-derived classes must match cfg.class_names (order matters for label indices)
assert train_ds.classes == cfg.class_names, (
    f"class_names mismatch.\nFound in data/train: {train_ds.classes}\n"
    f"Configured in cfg: {cfg.class_names}\nUpdate cfg.class_names to match exactly."
)

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))
print("Class → idx:", train_ds.class_to_idx)


In [ ]:

train_counts = Counter([label for _, label in train_ds.samples])
print("Train class distribution:")
for idx, name in enumerate(cfg.class_names):
    print(f"  {name}: {train_counts.get(idx, 0)}")

if cfg.use_weighted_sampler:
    class_sample_count = np.array([train_counts.get(i, 0) for i in range(cfg.num_classes)])
    class_weights = 1.0 / np.clip(class_sample_count, 1, None)
    sample_weights = np.array([class_weights[label] for _, label in train_ds.samples])
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, sampler=sampler,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
    )
else:
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
    )

val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)


In [ ]:

# Sanity check a batch — catches transform / label-mapping bugs before you burn GPU hours
imgs, labels = next(iter(train_loader))
print("Batch shape:", imgs.shape, "| Label range:", labels.min().item(), "-", labels.max().item())

def denormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(denormalize(imgs[i]).permute(1, 2, 0).numpy())
    ax.set_title(cfg.class_names[labels[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()


## Model

In [ ]:

def build_model(cfg: Config) -> nn.Module:
    if cfg.backbone == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if cfg.pretrained else None
        m = models.resnet50(weights=weights)
        in_features = m.fc.in_features
        m.fc = nn.Sequential(
            nn.Dropout(cfg.dropout),
            nn.Linear(in_features, cfg.num_classes),
        )
        backbone_params = [p for n, p in m.named_parameters() if not n.startswith("fc.")]
        head_params = list(m.fc.parameters())

    elif cfg.backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if cfg.pretrained else None
        m = models.efficientnet_b0(weights=weights)
        in_features = m.classifier[1].in_features
        m.classifier = nn.Sequential(
            nn.Dropout(cfg.dropout),
            nn.Linear(in_features, cfg.num_classes),
        )
        backbone_params = [p for n, p in m.named_parameters() if not n.startswith("classifier.")]
        head_params = list(m.classifier.parameters())

    elif cfg.backbone == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if cfg.pretrained else None
        m = models.convnext_tiny(weights=weights)
        in_features = m.classifier[2].in_features
        m.classifier[2] = nn.Sequential(
            nn.Dropout(cfg.dropout),
            nn.Linear(in_features, cfg.num_classes),
        )
        backbone_params = [p for n, p in m.named_parameters() if not n.startswith("classifier.2")]
        head_params = list(m.classifier[2].parameters())

    else:
        raise ValueError(f"Unknown backbone: {cfg.backbone}")

    m._backbone_params = backbone_params
    m._head_params = head_params
    return m


def set_backbone_trainable(model, trainable: bool):
    for p in model._backbone_params:
        p.requires_grad = trainable


model = build_model(cfg).to(cfg.device)
set_backbone_trainable(model, trainable=(cfg.freeze_backbone_epochs == 0))

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {n_total:,} total | {n_trainable:,} trainable")


## Loss, optimizer, scheduler

In [ ]:

# Class-weighted loss as a second lever against imbalance (works alongside the sampler;
# if you'd rather rely on only one mechanism, set use_weighted_sampler=False above and
# keep this, or vice versa — using both aggressively can over-correct).
class_sample_count = np.array([train_counts.get(i, 0) for i in range(cfg.num_classes)])
loss_weights = torch.tensor(
    (1.0 / np.clip(class_sample_count, 1, None)) * class_sample_count.sum() / cfg.num_classes,
    dtype=torch.float32,
).to(cfg.device)

criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=cfg.label_smoothing)

optimizer = torch.optim.AdamW(
    [
        {"params": model._backbone_params, "lr": cfg.lr_backbone},
        {"params": model._head_params, "lr": cfg.lr_head},
    ],
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.device == "cuda"))


## Train / validation loop

In [ ]:

def run_epoch(model, loader, criterion, optimizer=None, scaler=None, device="cuda", grad_clip=1.0):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss, all_preds, all_labels = 0.0, [], []
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                    logits = model(imgs)
                    loss = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)
            all_preds.append(logits.argmax(1).detach().cpu())
            all_labels.append(labels.detach().cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    avg_loss = total_loss / len(loader.dataset)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, bal_acc, macro_f1


In [ ]:

history = []
best_bal_acc = -1.0
epochs_no_improve = 0
best_ckpt_path = Path(cfg.output_dir) / "best_model.pt"

for epoch in range(1, cfg.epochs + 1):
    # staged unfreezing
    if epoch == cfg.freeze_backbone_epochs + 1:
        set_backbone_trainable(model, trainable=True)
        print(f"[epoch {epoch}] backbone unfrozen")

    t0 = time.time()
    train_loss, train_bal_acc, train_f1 = run_epoch(
        model, train_loader, criterion, optimizer, scaler, cfg.device, cfg.grad_clip_norm
    )
    val_loss, val_bal_acc, val_f1 = run_epoch(
        model, val_loader, criterion, optimizer=None, scaler=None, device=cfg.device
    )
    scheduler.step()
    dt = time.time() - t0

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_bal_acc": train_bal_acc,
        "train_f1": train_f1, "val_loss": val_loss, "val_bal_acc": val_bal_acc,
        "val_f1": val_f1, "lr_head": optimizer.param_groups[1]["lr"], "seconds": dt,
    })
    print(f"[{epoch:03d}/{cfg.epochs}] "
          f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"val_bal_acc={val_bal_acc:.4f} val_f1={val_f1:.4f} ({dt:.1f}s)")

    if val_bal_acc > best_bal_acc:
        best_bal_acc = val_bal_acc
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "cfg": cfg.__dict__,
            "epoch": epoch,
            "val_bal_acc": val_bal_acc,
        }, best_ckpt_path)
        print(f"  ↳ new best (val_bal_acc={val_bal_acc:.4f}), checkpoint saved")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= cfg.early_stopping_patience:
            print(f"Early stopping at epoch {epoch} (no improvement for {cfg.early_stopping_patience} epochs)")
            break

pd.DataFrame(history).to_csv(Path(cfg.output_dir) / "training_history.csv", index=False)
print(f"\nBest val balanced accuracy: {best_bal_acc:.4f} — checkpoint at {best_ckpt_path}")


In [ ]:

hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist_df.epoch, hist_df.train_loss, label="train")
axes[0].plot(hist_df.epoch, hist_df.val_loss, label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(hist_df.epoch, hist_df.train_bal_acc, label="train")
axes[1].plot(hist_df.epoch, hist_df.val_bal_acc, label="val")
axes[1].set_title("Balanced Accuracy"); axes[1].legend()

axes[2].plot(hist_df.epoch, hist_df.train_f1, label="train")
axes[2].plot(hist_df.epoch, hist_df.val_f1, label="val")
axes[2].set_title("Macro F1"); axes[2].legend()
plt.tight_layout()
plt.savefig(Path(cfg.output_dir) / "training_curves.png", dpi=150)
plt.show()


## Evaluation on held-out test set

In [ ]:

checkpoint = torch.load(best_ckpt_path, map_location=cfg.device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val_bal_acc={checkpoint['val_bal_acc']:.4f})")

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader):
        imgs = imgs.to(cfg.device)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)
        all_preds.append(probs.argmax(1).cpu())
        all_probs.append(probs.cpu())
        all_labels.append(labels)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs = torch.cat(all_probs).numpy()

print(classification_report(all_labels, all_preds, target_names=cfg.class_names, digits=3))
print("Balanced accuracy:", balanced_accuracy_score(all_labels, all_preds))


In [ ]:

cm = confusion_matrix(all_labels, all_preds, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(cfg.num_classes)); ax.set_xticklabels(cfg.class_names, rotation=45, ha="right")
ax.set_yticks(range(cfg.num_classes)); ax.set_yticklabels(cfg.class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized)")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(Path(cfg.output_dir) / "confusion_matrix.png", dpi=150)
plt.show()



## Fairness check across broad tone groups

Per-class accuracy alone can hide systematic bias — e.g. a model that's excellent on
light tones and mediocre on dark tones can still post a decent overall accuracy.
Group the fine-grained classes into **Light / Medium / Dark** (Monk 1–3 / 4–6 / 7–10)
and check that accuracy doesn't collapse for any group.


In [ ]:

# Adjust this mapping if your class_names / grouping differ from the default Monk 1-10 setup
def monk_group(idx):
    monk_level = idx + 1  # assumes class_names[i] corresponds to Monk level i+1
    if monk_level <= 3:
        return "Light"
    elif monk_level <= 6:
        return "Medium"
    else:
        return "Dark"

groups_true = np.array([monk_group(i) for i in all_labels])
groups_pred = np.array([monk_group(i) for i in all_preds])

print("Per-group accuracy (exact class match):")
for g in ["Light", "Medium", "Dark"]:
    mask = groups_true == g
    if mask.sum() == 0:
        continue
    acc = (all_preds[mask] == all_labels[mask]).mean()
    print(f"  {g:8s}: n={mask.sum():5d}  exact-class acc={acc:.3f}")

print("\nPer-group accuracy (broad group match, i.e. off-by-a-shade is OK):")
for g in ["Light", "Medium", "Dark"]:
    mask = groups_true == g
    if mask.sum() == 0:
        continue
    acc = (groups_pred[mask] == groups_true[mask]).mean()
    print(f"  {g:8s}: n={mask.sum():5d}  group-level acc={acc:.3f}")


## Inference on a single image

In [ ]:

from PIL import Image

def predict_image(image_path, model, transform, class_names, device):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1).squeeze(0).cpu().numpy()
    top_idx = int(probs.argmax())
    return {
        "predicted_class": class_names[top_idx],
        "confidence": float(probs[top_idx]),
        "all_probs": {class_names[i]: float(p) for i, p in enumerate(probs)},
    }

# Example:
# result = predict_image("path/to/some_face.jpg", model, eval_transform, cfg.class_names, cfg.device)
# print(json.dumps(result, indent=2))



## Production export

Two artifacts for deployment:
- **TorchScript** — if serving from a Python/PyTorch backend (TorchServe, custom FastAPI service).
- **ONNX** — if serving from a non-Python stack, or via ONNX Runtime for lower-latency CPU inference.


In [ ]:

model.eval()
example_input = torch.randn(1, 3, cfg.img_size, cfg.img_size).to(cfg.device)

# --- TorchScript ---
traced = torch.jit.trace(model, example_input)
ts_path = Path(cfg.output_dir) / "model_traced.pt"
traced.save(str(ts_path))
print(f"TorchScript model saved to {ts_path}")

# --- ONNX ---
onnx_path = Path(cfg.output_dir) / "model.onnx"
torch.onnx.export(
    model, example_input, str(onnx_path),
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print(f"ONNX model saved to {onnx_path}")

# Sanity-check the ONNX export matches the PyTorch model's output
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    onnx_out = sess.run(None, {"input": example_input.cpu().numpy()})[0]
    with torch.no_grad():
        torch_out = model(example_input).cpu().numpy()
    max_diff = np.abs(onnx_out - torch_out).max()
    print(f"Max abs diff between PyTorch and ONNX outputs: {max_diff:.6f}")
    assert max_diff < 1e-3, "ONNX export diverges from PyTorch model — investigate before deploying"
except ImportError:
    print("onnxruntime not installed — skipping export verification (pip install onnxruntime)")


In [ ]:

# Save everything needed to reproduce/serve this model
with open(Path(cfg.output_dir) / "config.json", "w") as f:
    json.dump(cfg.__dict__, f, indent=2)

with open(Path(cfg.output_dir) / "class_names.json", "w") as f:
    json.dump(cfg.class_names, f, indent=2)

with open(Path(cfg.output_dir) / "requirements.txt", "w") as f:
    f.write(
        "torch>=2.1\n"
        "torchvision>=0.16\n"
        "scikit-learn>=1.3\n"
        "pandas>=2.0\n"
        "numpy>=1.24\n"
        "matplotlib>=3.7\n"
        "pillow>=10.0\n"
        "tqdm>=4.65\n"
        "onnx>=1.15\n"
        "onnxruntime>=1.16\n"
    )

print("Saved config.json, class_names.json, requirements.txt to", cfg.output_dir)



## Notes for taking this further into production

- **Bias/fairness monitoring**: track the per-group accuracy check above as a required
  gate in CI before promoting any new checkpoint — don't just gate on overall accuracy.
- **Data drift**: skin-tone appearance shifts with camera sensor, white balance, and
  lighting conditions much more than most CV tasks. Log input-image statistics
  (mean brightness/color) in production and alert on distribution shift.
- **Calibration**: consider temperature scaling on the validation set before deploying —
  raw softmax confidences from CNNs are typically overconfident.
- **Human review loop**: for anything consumer-facing, route low-confidence predictions
  (e.g. top-1 prob < 0.5) to a fallback / human-reviewable path rather than forcing a label.
- **Versioning**: pin the checkpoint, config.json, and class_names.json together as one
  deployable unit — a config/class-index mismatch is the most common silent production bug.
